In [7]:
import cv2

def detect_square(contours):
    best_contour = None
    best_area = 0

    for contour in contours:
        area = cv2.contourArea(contour)

        # Ignore les petits contours
        if area < 1000:
            continue

        perimeter = cv2.arcLength(contour, True)

        approx = cv2.approxPolyDP(
            contour,
            0.02 * perimeter,
            True
        )

        # On garde seulement les formes à 4 coins
        if len(approx) == 4:
            x, y, w, h = cv2.boundingRect(approx)

            ratio = w / h

            # Vérifie que c'est presque carré
            if 0.8 <= ratio <= 1.2:
                if area > best_area:
                    best_area = area
                    best_contour = approx

    return best_contour, best_area

In [ ]:
import cv2

#Read image
path = r"C:\Users\willi\ETE26\log635\labo2\EnsembleA_H2024\EnsembleA_H2024\EnsembleA_A2023\Cercles\Cercle2\4_Cercle2.png"
image = cv2.imread(path)

#Convert to grayscale
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

#Bilateral Filter
bilateral = cv2.bilateralFilter(gray, 9, 75, 75)

#Detect 
edges = cv2.Canny(bilateral, 50, 150)


cv2.imwrite("edges.jpg", edges)

# Find edges
contours, _ = cv2.findContours(
    edges,
    cv2.RETR_EXTERNAL,
    cv2.CHAIN_APPROX_SIMPLE
)

best_contour = None
best_area = 0



best_contour, best_area = detect_square(contours)

if best_contour is None:
    print("Aucun carré détecté")
else:
    x, y, w, h = cv2.boundingRect(best_contour)

    # Extraire la région d'intérêt
    roi = image[y:y+h, x:x+w]

    # 1. Redimensionner en 40x40
    roi = cv2.resize(roi, (40, 40))

    # 2. Mettre en gris
    roi_gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)

    # 3. Appliquer un flou
    roi_blur = cv2.GaussianBlur(roi_gray, (5, 5), 0)

    # Sauvegarder l'image finale
    cv2.imwrite("roi_final.jpg", roi_blur)


In [9]:
import cv2
import os

input_root = r"C:\Users\willi\ETE26\log635\labo2\EnsembleA_H2024\EnsembleA_H2024\EnsembleA_A2023"
output_root = r"C:\Users\willi\ETE26\log635\labo2\Ensemble_B"

classes = [
    r"Cercles\Cercle2",
    r"Cercles\Cercle5",
    r"Diamants\Diamant2",
    r"Diamants\Diamant5",
    r"Hexagones\Hexagone2",
    r"Hexagones\Hexagone5",
    r"Triangles\Triangle2",
    r"Triangles\Triangle5"
]

def detect_square(contours):
    best_contour = None
    best_area = 0

    for contour in contours:
        area = cv2.contourArea(contour)

        if area < 100:
            continue

        perimeter = cv2.arcLength(contour, True)
        approx = cv2.approxPolyDP(contour, 0.05 * perimeter, True)

        if len(approx) == 4:
            x, y, w, h = cv2.boundingRect(approx)
            ratio = w / h

            if 0.7 <= ratio <= 1.3 and area > best_area:
                best_area = area
                best_contour = approx

    return best_contour, best_area


def process_image(input_path, output_path):
    image = cv2.imread(input_path)

    if image is None:
        return False

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    bilateral = cv2.bilateralFilter(gray, 9, 75, 75)
    edges = cv2.Canny(bilateral, 50, 150)

    contours, _ = cv2.findContours(
        edges,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    best_contour, best_area = detect_square(contours)

    if best_contour is None:
        return False

    x, y, w, h = cv2.boundingRect(best_contour)

    roi = image[y:y+h, x:x+w]
    roi = cv2.resize(roi, (40, 40))
    roi_gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    roi_blur = cv2.GaussianBlur(roi_gray, (5, 5), 0)

    cv2.imwrite(output_path, roi_blur)
    return True


for class_path in classes:
    input_folder = os.path.join(input_root, class_path)

    class_name = os.path.basename(class_path)
    output_folder = os.path.join(output_root, class_name)

    os.makedirs(output_folder, exist_ok=True)

    for filename in os.listdir(input_folder):
        if filename.lower().endswith((".png", ".jpg", ".jpeg")):
            input_path = os.path.join(input_folder, filename)
            output_path = os.path.join(output_folder, filename)

            process_image(input_path, output_path)


In [ ]:
import os
import random
import shutil

source_root = r"C:\Users\willi\ETE26\log635\labo2\Ensemble_B"
dest_root = r"C:\Users\willi\ETE26\log635\labo2\Ensemble_B_20"

NB_IMAGES = 20

random.seed(42) 

for class_name in os.listdir(source_root):

    source_folder = os.path.join(source_root, class_name)

    if not os.path.isdir(source_folder):
        continue

    dest_folder = os.path.join(dest_root, class_name)
    os.makedirs(dest_folder, exist_ok=True)

    images = [
        f for f in os.listdir(source_folder)
        if f.lower().endswith((".png", ".jpg", ".jpeg"))
    ]

    selected = random.sample(
        images,
        min(NB_IMAGES, len(images))
    )

    for image_name in selected:
        shutil.copy2(
            os.path.join(source_folder, image_name),
            os.path.join(dest_folder, image_name)
        )

    print(f"{class_name}: {len(selected)} images copiées")

print("Terminé")